# ▂▂▂▂▂▂▂▂▂▂▂▂

#**Impact of Query vs Output Subspaces**

#S1. Overview

Author: Chris McCormick

TL;DR:

1. I ran some experiments at closer to GPT-2 scale.
    * Output subspace still makes things slower.
    * Had issues with exploding gradients that I still need to resolve.
        * Worked around them by reducing the layer count.
2. I compared "just a query subspace" and "just an output subspace" and found that they have _very similar impacts_!
3. Next steps are:
    * Resolve the gradient problem
    * Re-run the above experiments at a scale which Koko finds, where subspace actually delivers speed-up.
        * If it turns out to be huge, maybe we'll just run at GPT-2-small to assess downstream performance, and show Koko's results for assessing throughput at larger scales.

# S2. GPT-2-small Training Runs

I decided to try a few training runs of the DeepSeekV3 model (in `subspace_decoder/`) at the scale of GPT-2-small (12 layers, d_model = 768, 12 heads of size 64) because:

1. It seems like a good scale to do some more thorough benchmarking at (and be able to compare to other models / papers).
2. I hoped that this size would finally show a speed improvement from the latent spaces.
3. I wanted to do some additional runs to compare the impact of the query latent vs. the output latent, to see if one was more beneficial / harmful than the other.

Here's what I learned.


## 2.1. Switching to Dense FFNs

The DeepSeekV3 implementation includes support for replacing MoE layers with dense ones, and you can use this to set the entire model to use dense FFNs.

I tried this, and the results seemed like a strong improvement on training. (I didn't do a full run for comparison, just enough to confirm that it was faster). I don't appear to have recorded measurements on this, that I can find.

However, it seemed like a reasonable choice regardless since the attention latent spaces are what we're focusing on.



## 2.2. Exploding Gradients

When I tried the GPT-2 scale parameters, training failed. Within 50 steps the loss was a massive number and the grad norm was "NaN".


<img src='https://lh3.googleusercontent.com/d/1-m1XL_j06Y36mO8skz_y19WGoIM3wVbr' alt='Training loss and grad norm for a failed run' width='900' />

**What didn't work...**

I asked GPT for advice and tried its various suggestions, none of which resolved it:

* Dropped learning rate dramatically from 5e-4 to 1e-5.
* Set max_grad_norm = 1.0
* Switched to 32-bit precision.
* Disabled torch.compile
* Switched from Flash Attention 2 to PyTorch SDPA.
* Increase RMSNorm epsilon from 1e-6 to 1e-5.*
    
> *On RMSNorm, the DeepSeekV3 config parameter only applies to the layer input and attention output norms, and _not_ to the query and key-value latent space norms. It's possible that this was the root cause--I'll come back to this further down.

**What worked**

None of the above changes were necessary. The deciding factor was some combination of **d_model** and the number of **layers**.

I changed both at the same time--dropping to 8 layers and d_model=512, and that worked.

I creeped back up to **d_model = 576** and **10 layers**. This configuration trained fine and it's where I ended up.

If I increased to 12 layers, it would cause the blow-up (as in the plots above).

> I didn't try d_model = 768 at 10 layers, so I don't actually know whether d_model was a contributing factor or not--only that going to 12 layers was enough to break it.


**Final Configuration**

I ran my experiments with:

```python
    "hidden_size": 576,
    "num_hidden_layers": 10,
    "num_attention_heads": 12,
    "v_head_dim": 48,
```

And for the latent spaces:
    
```python
    "q_lora_rank": 192,
    "kv_lora_rank": 96,
    "o_latent_dim": 192,
```
    





## 2.3. Impact of Query vs. Output Latent

So far, our experience has been that adding the output subspace has no benefit: it slows things down and decreases benchmark accuracy.

I wanted to find out whether these negative effects were unique to the output space, or if this was _true of the query space_ as well.


**Impact on Throughput**

Assuming the subspaces are the same size (which they are for us), the query latent space and the output latent space have the same affect in terms of reducing the number of parameters and the number of operations.

The order of operations is different, though, so it's conceivable that PyTorch could take better advantage of one decomposition versus the other.

**Impact on Accuracy**

What's less clear is whether they have different impacts on "model quality" / downstream performance.

Does the Output space fundamentally need higher rank than the Query space, because of some unique aspect of its role?

#### Results


I pre-trained and fine-tuned three models. All three used a kv-latent space, but they varied in whether they included a query or output space (or neither).

The margins are very small, but in both pre-training and fine-tuning, adding a **query** subspace had the larger negative impact on runtime.

Adding the query subspace had the lowest perplexity on the pre-training validation data, and the highest accuracy on the SST-2 validation data, but lower test accuracy (suggesting it may have overfit).

_Parameter Count_

| Experiment   | Parameter Count    |
|--------------|--------------------|
| Neither      | 73.23M             |
| Output Only  | 72.18M (-1.05M)    |
| Query Only   | 72.18M (-1.05M)    |

<br/>

_Pre-Training_

| Experiment   | Runtime            | Perplexity        |
|--------------|--------------------|-------------------|
| Neither      | **57m 15s**        | 28.876            |
| Output Only  | 57m 37s (+0m 22s)  | 28.878 (+0.002)   |
| Query Only   | 58m 28s (+1m 13s)  | **28.152** (−0.724)   |


_Fine-Tuning Results_

| Experiment   | Runtime           | test/accuracy     | eval/accuracy     |
|--------------|-------------------|-------------------|-------------------|
| Neither      | **8m 12s**            | **87.615**            | 94.402            |
| Output Only  | 8m 14s (+2s)      | **87.615** (+0.000)   | 94.254 (−0.148)   |
| Query Only   | 8m 18s (+6s)      | 87.271 (−0.344)   | **94.506** (+0.104)   |





**RMSNorm**

There's an important distinction that we'll need to resolve--I've found in other experiments that adding an RMSNorm in between the decomposed output matrices (in the same way they do on the query and kv decompositions) can be beneficial. Here, however, the RMSNorm was causing exploding gradients, so I removed it.

That might explain the differences between the two results.

**Interpretation**

I'm pretty excited by this because I think it serves as strong evidence that whatever is true of the query-latent projection may also be true of the output-latent projection.


# S3. Next Steps


**1. RMSNorm**

I think we should resolve the RMSNorm issue, and then repeat the above experiments with a longer sequence length and a larger variety of benchmarks to see if the similarity holds up with longer context and different tasks.

Ideally, at a scale where the decompositions start to provide a speed-up rather than a slowdown!



**2. Broader benchmarks**

One paper I read recently called MoLAE did some similar GPT-2-small scale experiments, looking at subspaces for Mixture of Experts layers. Looking at the list benchmarks they evaluated on could be a good reference.



**3. Dataset diversity**

Currently, I'm training on wikitext103, which has 103M tokens, and doing that for something like 15 epochs?

So the model's seeing the same data over and over.

Especially as we scale the model up, we should probably switch to something bigger. I'm thinking 1% of C4?




### 3.1. Experiment Ideas

_Efficiently train on longer context_

* Train at short, then final steps longer, then use RoPE to further expand.



_Joint subspaces_

* Try joint query-output subspace.
    * I wouldn't really expect this to work in the same layer, but we could try it anyway.
    * More interesting would be to look at connecting the output subspace to either:
        * An input subspace for the FFN
        * The query subspace of the next attention block


### 3.2. New GitHub Issues


1. Separate out the sft configs from the model+pre-train configs.
    * The current approach of specifying all three together is brittle and adds extra overhead to defining new runs.
    * Instead, have finetune_sst2.py take in a path to:
        * A separate sft config script (which can probably be re-used across all runs!)
        * The path to the config file for the pre-trained model.
    * Have finetune_sst2.py auto-complete the fields such as the run name and checkpoint folder from those two.



# ▂▂▂▂▂▂▂▂▂▂▂▂

# Appendix - Lessons Learned

---

## 1. Python Environment



I'm new to wrangling with package dependencies--for a long time now I've relied on Google Colab to magically resolve all of that for me.

I got the impression that Lambda would be the same way, because the instances come pre-loaded with "Lambda Stack", but it's been a huge headache, especially as someone who hasn't had to learn how to deal with these issues.

Here's where I think the difference lies:

The instance comes with compatible versions of CUDA and pytorch installed (and flash attention, which is cool!), but nothing from huggingface. `transformers` and `datasets` seem to be tricky to get setup cleanly.



**Disabling Tensorflow**

The biggest issue I had, and couldn't get around cleanly, was that it wants to pull in tensorflow, and `tf-keras`, even though I'm not using those.

The workaround GPT gave me was to put multiple flags and checks at the top of my scripts to try and prevent the tensorflow references:

```python
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"              # older check some codepaths still honor
# Optional: if Keras 3 is on the system and ever gets touched, force non-TF backend
os.environ.setdefault("KERAS_BACKEND", "torch")

from transformers.utils import is_tf_available
print("TF available (Transformers thinks):", is_tf_available())  # should be False
```

(Fortunately?) that did work, and even resolved some ugly warnings that would get dumped out when I imported transformers.



**Pinning Versions**

The other thing I'm doing is just specifying exact versions of the dependencies which are working for me:

```bash
pip3 install --user -U \
  "transformers==4.56.0" \
  "datasets==4.0.0" \
  "accelerate==1.10.1" \
  "deepspeed==0.17.5" \
  "wandb==0.21.3" \
  "tqdm==4.67.1" \
  "pyarrow==21.0.0" \
  "huggingface-hub==0.34.4" \
  "safetensors==0.6.2" \
  "tokenizers==0.22.0"
```

There's a setup script now at `infra/lambda/setup_lambda.sh`

---

## 2. Working with Agents

The Agents are great at writing professional code, but their style is better suited to product building than research and rapid prototyping.

At some point, while super fed-up with this, I wrote the below "style guide" / rant to give to the agents as context.

```
# ============================================================================
# IMPORTANT: STYLE GUIDE
# This is a **prototype**. Don't polish it prematurely.
#
# This means:
# - Prioritize legibility over re-use and robustness.
#     - DO NOT use get_attr. All settings are required. If something is missing,
#       that was a programmer error, and we want the code to **crash**.
#     - Do not clutter the code with "raise ValueError", that's premature.
#     - We can add those things in later when the code is mature.
#
# Minimize boiler plate and safety checks.
# Don't create silent bugs by implementing fallbacks.
# Comment heavily.
# Avoid packing operations into dense, single lines.
# Prefer flat code to small helper functions. Factorization is for mature code,
# not prototypes. It requires the developer to invest time and energy into
# becoming familiar with what the function does and how to use it.
#
# Note that some of the existing code is agent-written and doesn't currently
# follow these guidelines.
# ============================================================================
```

## 3. Workflow

Since my last journal:

1. I've been using the "runs" table in wandb and that's been **huge**.
2. The diff tool inside of Cursor has been helpful.

Some things to carry forward still:

1. **Connect Cursor to Lambda** - Letting the Agent execute code on the instance would be highly valuable; it could actually run its test cases. (See my [notes](https://colab.research.google.com/drive/1T0Js6Jz9Zm_YXDLn6Vszda0LUIm8-s9P#scrollTo=tscl1c8jNsvc) in the last journal)
2. **Delete last wandb run** - A cell in the "run experiment" notebooks which simply erases the most recent wandb run would be helpful. I end up with lots of dummy runs due to crashes or short tests.

